In [4]:
import json
import pandas as pd
import geopandas as gpd

renewables_path = "dashboard_data/renewables16.json"
precincts_path = "dashboard_data/precincts.geojson"
output_csv = "renewables_precinct_join.csv"

def find_first_existing_column(df, candidates, label):
    for col in candidates:
        if col in df.columns:
            return col
    raise KeyError(f"Could not find a column for {label}. Available columns:\n{list(df.columns)}")

def extract_records_from_json(obj):
    for k, v in obj.items():
        if isinstance(v, list):
            return v, [k]
    for k1, v1 in obj.items():
        if isinstance(v1, dict):
            for k2, v2 in v1.items():
                if isinstance(v2, list):
                    return v2, [k1, k2]
    raise ValueError("Could not find the list of plant records in the JSON.")

with open(renewables_path, "r") as f:
    raw = json.load(f)

print("Top-level keys:", list(raw.keys()))

records, record_path = extract_records_from_json(raw)
print("Found plant records at:", " -> ".join(record_path))
print("Number of raw records found:", len(records))

plants_df = pd.DataFrame(records)

print("\nPlant columns:")
print(plants_df.columns.tolist())
print("\nFirst 5 plant rows:")
print(plants_df.head())

plant_name_col = find_first_existing_column(
    plants_df,
    ["plant_name", "plantName", "name", "plant-name"],
    "plant name"
)

tech_col = find_first_existing_column(
    plants_df,
    ["technology", "technology_description", "technologyDescription", "tech", "energy_source", "fuel_type"],
    "technology type"
)

capacity_col = find_first_existing_column(
    plants_df,
    ["nameplate-capacity-mw", "nameplate_capacity", "nameplate_capacity_mw", "nameplateCapacity", "capacity_mw", "capacity"],
    "nameplate capacity"
)

county_col = find_first_existing_column(
    plants_df,
    ["county", "county_name", "countyName"],
    "county"
)

lat_col = find_first_existing_column(
    plants_df,
    ["latitude", "lat"],
    "latitude"
)

lon_col = find_first_existing_column(
    plants_df,
    ["longitude", "lon", "lng"],
    "longitude"
)

print("\nUsing plant columns:")
print("plant name:", plant_name_col)
print("technology:", tech_col)
print("capacity:", capacity_col)
print("county:", county_col)
print("latitude:", lat_col)
print("longitude:", lon_col)

plants_df[lat_col] = pd.to_numeric(plants_df[lat_col], errors="coerce")
plants_df[lon_col] = pd.to_numeric(plants_df[lon_col], errors="coerce")

before_drop = len(plants_df)
plants_df = plants_df.dropna(subset=[lat_col, lon_col]).copy()
after_drop = len(plants_df)

print(f"\nDropped {before_drop - after_drop} rows with missing/invalid coordinates.")
print(f"Remaining plant rows: {after_drop}")

plants_gdf = gpd.GeoDataFrame(
    plants_df,
    geometry=gpd.points_from_xy(plants_df[lon_col], plants_df[lat_col]),
    crs="EPSG:4326"
)

precincts_gdf = gpd.read_file(precincts_path)

print("\nPrecinct columns:")
print(precincts_gdf.columns.tolist())
print("\nPrecinct CRS:", precincts_gdf.crs)

if precincts_gdf.crs is None:
    precincts_gdf = precincts_gdf.set_crs("EPSG:4326")
elif precincts_gdf.crs.to_string() != "EPSG:4326":
    precincts_gdf = precincts_gdf.to_crs("EPSG:4326")

print("Precinct CRS after check:", precincts_gdf.crs)
print("Plant CRS:", plants_gdf.crs)

precinct_name_col = find_first_existing_column(
    precincts_gdf,
    ["precinct_name", "PRECINCT", "name", "Name", "PCTNAME", "precinct"],
    "precinct name"
)

congress_col = find_first_existing_column(
    precincts_gdf,
    ["congress_dist", "congressional_dist", "congressional_district", "CONG_DIST", "USCD", "cd", "district_1", "congress"],
    "US Congressional district"
)

senate_col = find_first_existing_column(
    precincts_gdf,
    ["senate_dist", "senate_district", "SEN_DIST", "senate", "sd"],
    "State Senate district"
)

house_col = find_first_existing_column(
    precincts_gdf,
    ["house_dist", "house_district", "HOUSE_DIST", "house", "hd"],
    "State House district"
)

print("\nUsing precinct columns:")
print("precinct name:", precinct_name_col)
print("congressional district:", congress_col)
print("senate district:", senate_col)
print("house district:", house_col)

possible_precinct_county_col = None
for c in ["county", "county_name", "COUNTY", "County"]:
    if c in precincts_gdf.columns:
        possible_precinct_county_col = c
        break

if possible_precinct_county_col:
    print("precinct county field found:", possible_precinct_county_col)
else:
    print("No separate county field found in precinct file.")

joined = gpd.sjoin(
    plants_gdf,
    precincts_gdf,
    how="left",
    predicate="within"
)

plants_df[capacity_col] = pd.to_numeric(plants_df[capacity_col], errors="coerce")
joined[capacity_col] = pd.to_numeric(joined[capacity_col], errors="coerce")

print("\nJoined columns:")
print(joined.columns.tolist())
print("\nJoined row count:", len(joined))

final_df = pd.DataFrame({
    "plant_name": joined[plant_name_col],
    "technology_type": joined[tech_col],
    "nameplate_capacity_mw": joined[capacity_col],
    "county": joined[county_col],
    "precinct_name": joined[precinct_name_col],
    "us_congressional_district": joined[congress_col],
    "state_senate_district": joined[senate_col],
    "state_house_district": joined[house_col]
})

final_df = final_df.reset_index(drop=True)

print("\nFinal table preview:")
print(final_df.head())

print("\nFinal row count:", len(final_df))

final_df.to_csv(output_csv, index=False)
print(f"\nCSV saved as: {output_csv}")

matched_count = final_df["precinct_name"].notna().sum()
unmatched_count = final_df["precinct_name"].isna().sum()

print("\n--- QUESTION CHECKS ---")
print(f"Plants matched to a precinct: {matched_count}")
print(f"Plants not matched: {unmatched_count}")

coord_counts = plants_df.groupby([lat_col, lon_col]).size().reset_index(name="count")
shared_coords = coord_counts[coord_counts["count"] > 1].sort_values("count", ascending=False)

print(f"\nNumber of unique coordinate pairs shared by more than one plant: {len(shared_coords)}")
if len(shared_coords) > 0:
    print("Examples of shared coordinates:")
    print(shared_coords.head(10))

if possible_precinct_county_col:
    county_compare = joined[[county_col, possible_precinct_county_col]].copy()
    county_compare = county_compare.dropna()

    county_compare["plant_county_clean"] = county_compare[county_col].astype(str).str.strip().str.lower()
    county_compare["precinct_county_clean"] = county_compare[possible_precinct_county_col].astype(str).str.strip().str.lower()

    county_mismatches = county_compare[
        county_compare["plant_county_clean"] != county_compare["precinct_county_clean"]
    ]

    print(f"\nCounty mismatches using precinct county field: {len(county_mismatches)}")
    if len(county_mismatches) > 0:
        print(county_mismatches.head(10))
else:
    print("\nNo explicit county field in precinct file, so county agreement may need to be discussed conceptually.")

Top-level keys: ['response', 'request', 'apiVersion', 'ExcelAddInVersion']
Found plant records at: response -> data
Number of raw records found: 106

Plant columns:
['period', 'stateid', 'stateName', 'sector', 'sectorName', 'entityid', 'entityName', 'plantid', 'plantName', 'generatorid', 'technology', 'energy_source_code', 'energy-source-desc', 'prime_mover_code', 'balancing_authority_code', 'balancing-authority-name', 'status', 'statusDescription', 'county', 'latitude', 'longitude', 'nameplate-capacity-mw', 'operating-year-month', 'unit', 'nameplate-capacity-mw-units']

First 5 plant rows:
    period stateid stateName            sector        sectorName entityid  \
0  2016-12      IA      Iowa  electric-utility  Electric Utility    14201   
1  2016-12      IA      Iowa  electric-utility  Electric Utility    14201   
2  2016-12      IA      Iowa  electric-utility  Electric Utility      309   
3  2016-12      IA      Iowa       ipp-non-chp       IPP Non-CHP    59496   
4  2016-12      I